# Lab 02: Quick Wins

## Overview

In this notebook, we apply **low-effort** optimizations that tighten the agent's responses and trim output tokens. The key insight: **prompt structure matters more than length**. A clearer prompt and a couple of model-config flags are the cheapest improvements you can make before reaching for anything heavier.

**What you'll learn:**
- How to structure system prompts for clarity and consistency
- How to use `max_tokens` to cap output length
- How to set an appropriate `temperature` for accuracy

**Optimizations in this notebook:**
1. Well-structured prompt — sections, numbered guidelines, few-shot examples (restructured, not shortened)
2. `max_tokens=1024` to bound output length
3. `temperature=0.1` for accurate, consistent support responses

> The lever here is **output**, not input. Restructuring + `max_tokens` cut what the model *generates* per reply; the system prompt itself doesn't get shorter in this lab (that's what caching in Lab 03 is for). Watch the output-token column — cost and latency follow it, but the bigger cost wins come in later labs.

## Prerequisites

- Completed Lab 01 (baseline agent deployed)
- Baseline metrics recorded

## Workshop Journey

```
01 Baseline → [02 Quick Wins] → 03 Caching → 04 Routing → 05 Guardrails → 06 Skills + Gateway → 07 Evaluations
                   ↑
              You are here
```

## Step 1: Setup

In [1]:
from __future__ import annotations

import json
import os
import uuid
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from bedrock_agentcore_starter_toolkit import Runtime

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
control_client = boto3.client("bedrock-agentcore-control", region_name=region)
data_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_runtime = Runtime()

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'Not set')}")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Region: us-east-1
Langfuse Host: https://d1fnzg75c19u2d.cloudfront.net


## Step 2: Review the Quick Wins Optimizations

Let's compare the baseline vs optimized configurations.

## Optimization 1: Well-Structured System Prompt

### Baseline Prompt (v1)

In [2]:
from agents.v1_baseline import SYSTEM_PROMPT as BASELINE_PROMPT

print(BASELINE_PROMPT)


You are a customer support assistant and your job is to try to help customers as best
you can with whatever they need. You work for TechMart Electronics which is a company
that sells electronics and technology products. Please try to be helpful and friendly
and professional and also knowledgeable and empathetic and patient and understanding
too if you can. TechMart Electronics is a retailer that sells things like consumer
electronics and computers and laptops and smartphones and mobile phones and tablets
and also audio equipment and headphones and speakers and smart home devices and gaming
consoles and gaming accessories and various other products and accessories that are
related to technology and electronics. Can you please help customers with their
questions and concerns and issues and problems and inquiries about products and
services and policies and returns and technical stuff and troubleshooting and other
things they might need help with or want to know about?

Please try your b

### Opportunities for Improvement

| Area | Observation |
|------|-------------|
| **Structure** | Dense paragraphs without visual hierarchy make it difficult for the model to quickly locate relevant instructions |
| **Hedging language** | Phrases like "try to", "as best you can", "hopefully", "if possible" introduce ambiguity about expected behavior |
| **Filler phrases** | "Please", "Can you please", "It would be great if" add tokens without providing actionable guidance |
| **Task definition** | The expected output format and response structure are not specified |
| **Redundancy** | Adjective chains like "helpful and friendly and professional and also knowledgeable and empathetic" could be condensed |

The fix isn't to make it *shorter* — it's to make it *clearer*: define the output format, replace hedging with direct instructions, and show the model what good looks like with examples.

---

### Optimized Prompt (v2)

In [3]:
from utils.agent_config import SYSTEM_PROMPT_TEXT as OPTIMIZED_PROMPT

print(OPTIMIZED_PROMPT)


# ROLE

You are Alex, a customer support specialist at TechMart Electronics, a leading
retailer of consumer electronics including computers, smartphones, tablets, audio
equipment, smart home devices, and gaming products. Your role is to help customers
with product information, returns and policies, and technical support. Be friendly,
accurate, and solution-focused in all interactions.

# RESPONSE FORMAT

Always structure your response with these three fields:

- **answer**: Clear, helpful response to the customer. Use bullet points for lists,
  numbered steps for instructions, and include specific details like prices, return
  windows, and specifications.
- **category**: Classify as "product" (info, recommendations), "policy" (returns,
  warranties), "technical" (troubleshooting, setup), or "general" (greetings, other)
- **confidence**: "high" (verified with tools), "medium" (partial info), or "low"
  (uncertain, recommend escalation)

# GUIDELINES

1. Always use tools to verify infor

### What Makes This Prompt Effective

| Technique | Implementation |
|-----------|----------------|
| **Visual hierarchy** | Headers (`# ROLE`, `# GUIDELINES`, `# EXAMPLES`) organize content into scannable sections |
| **Direct language** | "You are Alex" instead of "Please try to be..."; action-oriented guidelines |
| **Numbered guidelines** | Clear, prioritized instructions (1–6) the model can reference |
| **Explicit output format** | Defines required fields: answer, category, confidence |
| **Few-shot examples** | 4 examples demonstrate tool usage, response structure, and expected behavior |

**Key insight:** few-shot examples are one of the most effective prompting techniques. They show the model exactly what you want — tool-calling patterns, output structure, tone — without verbose explanation.

**A note on length:** the optimized prompt *is* somewhat shorter than the baseline (the baseline rambles — it repeats "try to" 22 times and piles on redundant adjective chains). But the win this lab delivers is **not** from those saved prompt tokens. It's still a large prompt — it carries the full troubleshooting method and four worked examples — and once you factor in tool definitions and the per-request overhead, **total input tokens stay roughly flat** in the comparison below (they even tick up slightly). The real savings come from the **output** side: a clear output contract plus `max_tokens` make replies tighter. Shrinking the *input* cost is Lab 03's job (caching), which is why we kept the prompt comfortably above the ~1,024-token floor caching needs.

---

### Further Reading

For more on prompt engineering best practices, see:

- [Anthropic: Prompt Engineering Overview](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview)
- [AWS: Prompt Engineering Techniques with Claude on Amazon Bedrock](https://aws.amazon.com/blogs/machine-learning/prompt-engineering-techniques-and-best-practices-learn-by-doing-with-anthropics-claude-3-on-amazon-bedrock/)
- [OpenAI: Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic: Use XML Tags to Structure Your Prompts](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/use-xml-tags)

## Optimizations 2 & 3: Model Configuration

### Optimization 2: `max_tokens=1024`

Caps output length to prevent runaway responses. 1024 tokens is enough for detailed troubleshooting while keeping cost predictable. It's also the safety net for output length — the model stops on its own (`stop_reason: "end_turn"`) when it's done, and `max_tokens` bounds the worst case.

### Optimization 3: `temperature=0.1`

Low temperature for accuracy and consistency. Customer support needs factual, predictable responses — not creative variation.

| Temperature | Use Case |
|-------------|----------|
| 0.0 – 0.3 | Factual tasks, customer support, code generation |
| 0.4 – 0.7 | Balanced creativity and accuracy |
| 0.8 – 1.0 | Creative writing, brainstorming |

### Final Model Configuration

```python
model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-6",
    temperature=0.1,    # Optimization 3
    max_tokens=1024,    # Optimization 2
)
```

> **What about `stop_sequences`?** Bedrock supports them — the server halts generation the moment the output matches one of your strings — but they're a sharp edge for a chat agent. An earlier version of this workshop used `stop_sequences=["###", "END_RESPONSE"]`; `###` is also a Markdown heading, so the server cut answers off mid-reply the instant the model wrote a heading. We dropped them entirely: `max_tokens` plus the model's own `end_turn` already bound output, so a stop string only adds truncation risk. If you ever do need one, pick a token that can **never** appear in normal output (e.g. `<<END>>`) and instruct the model to emit it.

In [4]:
# Review the v2 agent code
agent_file = Path("agents/v2_quick_wins.py")
print(agent_file.read_text())

"""
V2 Quick Wins Agent - Low-effort optimizations.
- Concise system prompt
- max_tokens limit
- Low temperature for accuracy
"""

from __future__ import annotations

import os

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry

from utils.agent_config import MODEL_SONNET, SYSTEM_PROMPT_TEXT, setup_langfuse_telemetry
from utils.tools import get_product_info, get_return_policy, get_technical_support, web_search

setup_langfuse_telemetry()

app = BedrockAgentCoreApp()

SYSTEM_PROMPT = SYSTEM_PROMPT_TEXT


@app.entrypoint
def invoke(payload):
    user_input = payload.get("prompt", "")

    telemetry = StrandsTelemetry()
    telemetry.setup_otlp_exporter()

    model = BedrockModel(
        model_id=MODEL_SONNET,
        temperature=0.1,
        max_tokens=1024,
        region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    )

    agent = Agent(
        model

## Step 3: Deploy the Quick Wins Agent

In [5]:
agent_name = "customer_support_v2_quick_wins"
agent_file = str(Path("agents/v2_quick_wins.py").absolute())
requirements_file = str(Path("requirements-for-agentcore.txt").absolute())

# Clean up any existing build files from previous labs
for f in ["Dockerfile", ".dockerignore", ".bedrock_agentcore.yaml"]:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Removed existing: {f}")

print(f"Configuring agent: {agent_name}")
agentcore_runtime.configure(
    entrypoint=agent_file,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=requirements_file,
    region=region,
    agent_name=agent_name,
)

Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/agents/v2_quick_wins.py, bedrock_agentcore_name=v2_quick_wins
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: customer_support_v2_quick_wins


Removed existing: Dockerfile
Removed existing: .dockerignore
Removed existing: .bedrock_agentcore.yaml
Configuring agent: customer_support_v2_quick_wins


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.dockerignore
Setting 'customer_support_v2_quick_wins' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/Dockerfile'), dockerignore_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='739907928487', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

In [6]:
# Modify Dockerfile for Langfuse
dockerfile_path = Path("Dockerfile")
if dockerfile_path.exists():
    content = dockerfile_path.read_text()
    # Replace opentelemetry-instrument wrapper with direct python call
    # Keep the correct module path using regex
    if "opentelemetry-instrument" in content:
        import re

        content = re.sub(
            r'CMD \["opentelemetry-instrument", "python", "-m", "([^"]+)"\]', r'CMD ["python", "-m", "\1"]', content
        )
        dockerfile_path.write_text(content)
        print("Dockerfile modified for Langfuse")
    else:
        print("Dockerfile already configured or using different format")
else:
    print("Dockerfile not found - will be created during deployment")

Dockerfile modified for Langfuse


In [7]:
env_vars = {
    "LANGFUSE_BASE_URL": os.environ.get("LANGFUSE_BASE_URL"),
    "LANGFUSE_PUBLIC_KEY": os.environ.get("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.environ.get("LANGFUSE_SECRET_KEY"),
    "PYTHONUNBUFFERED": "1",
}

print("Deploying to AgentCore Runtime...")
launch_result = agentcore_runtime.launch(env_vars=env_vars, auto_update_on_conflict=True)
agent_arn = launch_result.agent_arn
print(f"Agent deployed: {agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'customer_support_v2_quick_wins' to account 739907928487 (us-east-1)
Generated image tag: 20260602-171147-977
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_v2_quick_wins


Deploying to AgentCore Runtime...


ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v2_quick_wins
Getting or creating execution role for agent: customer_support_v2_quick_wins
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-449c4d3c64


✅ Reusing existing ECR repository: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v2_quick_wins


✅ Reusing existing execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-449c4d3c64
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-449c4d3c64
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: customer_support_v2_quick_wins
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-449c4d3c64
Reusing existing CodeBuild execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-449c4d3c64
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: customer_support_v2_quick_wins/source.zip
Updated CodeBuild project: bedrock-agentcore-customer_support_v2_quick_wins-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 1s)
✅ QUEUED completed in 1.6s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 7.9s
🔄 DO

Agent deployed: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v2_quick_wins-WwfeJI2Ll9


In [8]:
# Save the agent ARN for later use
agent_arn = launch_result.agent_arn
print(f"Agent ARN: {agent_arn}")

# Grant the auto-created execution role the shared runtime permissions
# (SSM KB lookup + Bedrock Retrieve + ApplyGuardrail) used across all labs.
from utils.runtime_helpers import ensure_runtime_permissions

ensure_runtime_permissions(region)

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v2_quick_wins-WwfeJI2Ll9
Granted customer-support runtime permissions to AmazonBedrockAgentCoreSDKRuntime-us-east-1-449c4d3c64


## Step 4: Test the Optimized Agent

Run the same test scenarios as the baseline to compare metrics.

In [9]:
def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))

In [10]:
# Import Langfuse metrics helper
from utils.langfuse_metrics import (
    clear_metrics,
    collect_metric,
    get_latest_trace_metrics,
    print_metrics,
    print_metrics_table,
)

# Clear any previously collected metrics
clear_metrics()

# Standard test prompts - each demonstrates a specific tool usage pattern
TEST_PROMPTS = [
    # Single tool: get_return_policy
    ("Return Policy", "What is your return policy for laptops?"),
    # Single tool: get_product_info
    ("Product Info", "Tell me about your smartphone options"),
    # Single tool: get_technical_support (Bedrock KB)
    ("Technical Support", "My laptop won't turn on, can you help me troubleshoot?"),
    # Multi-tool: get_product_info + get_return_policy
    ("Multi-part Question", "I want to buy a laptop. What are the specs and what's the return policy?"),
    # No tool: General greeting
    ("General Question", "Hello! What can you help me with today?"),
]

# Run all tests and collect metrics
for test_name, prompt in TEST_PROMPTS:
    print("=" * 60)
    print(f"Test: {test_name}")
    print("=" * 60)

    response = invoke_agent(prompt)
    print(response)

    # Fetch and collect metrics
    metrics = get_latest_trace_metrics(
        agent_name="customer-support-v2-quick-wins",
        wait_seconds=5,
        max_retries=5,
        timeout_seconds=120,
    )
    print_metrics(metrics, test_name)
    collect_metric(metrics, test_name)

Test: Return Policy
Here's a summary of our laptop return policy:

- **Return Window:** 30 days from the date of purchase
- **Condition Requirements:** Must be in original packaging with all accessories included and no physical damage
- **Restocking Fee:** None for defective items; 15% fee applies for change-of-mind returns
- **Return Shipping:** Free if the laptop is defective; customer covers shipping for change-of-mind returns
- **Refund Timeline:** 7–10 business days after the returned item is inspected
- **How to Return:** You can initiate a return through our **online RMA portal** or bring the laptop to any **TechMart store** in person
- **Warranty:** All laptops come with a **1-year manufacturer warranty**, and extended warranty options are also available

Would you like help starting a return, or do you have any other questions about our laptop policies or products?

- **category:** policy
- **confidence:** high

                    LANGFUSE METRICS
  Test:          Return Poli

In [11]:
# Print summary table
print_metrics_table()

# Save metrics for comparison in later notebooks
from utils.langfuse_metrics import save_metrics

save_metrics("v2")


                                  METRICS SUMMARY
               Test Latency    Cost Input Output Cache Read Tokens Cache Write Tokens
      Return Policy   6.34s $0.0238 6,541    277                 0                  0
       Product Info   8.78s $0.0254 6,588    379                 0                  0
  Technical Support   8.59s $0.0250 6,980    273                 0                  0
Multi-part Question  10.64s $0.0297 6,903    600                 0                  0
   General Question   4.85s $0.0123 3,171    183                 0                  0
---------------------------------------------------------------------------------------------------------
  TOTALS: Latency(avg): 7.84s | Cost: $0.1162 | Input: 30,183 | Output: 1,712
          Cache Read Tokens: 0 | Cache Write Tokens: 0

Metrics saved as 'v2' → .lab_metrics.json


## Step 5: Compare with v1 (Baseline)

Enter your metrics from Lab 01 (v1 baseline) to compare cost, latency, and token usage.

In [12]:
from utils.langfuse_metrics import load_metrics, print_comparison

# Load metrics from Lab 01 (saved automatically when you ran print_metrics_table())
v1 = load_metrics("v1")

# Or enter manually if Lab 01 metrics weren't saved:
# v1 = {"total_cost": 0.1042, "avg_latency": 8.90, "total_input_tokens": 24523, "total_output_tokens": 2040}

# Print comparison (current metrics auto-calculated from collected)
print_comparison(
    prev_name="v1 (Baseline)",
    curr_name="v2 (Quick Wins)",
    prev_cost=v1["total_cost"],
    prev_latency=v1["avg_latency"],
    prev_input_tokens=v1["total_input_tokens"],
    prev_output_tokens=v1["total_output_tokens"],
)

Loaded 'v1' metrics: cost=$0.1184, latency=8.38s, input=29,415, output=2,009
  V1 (BASELINE) vs V2 (QUICK WINS) COMPARISON
Metric                    v1 (Baseline)    v2 (Quick Wins)       Change
----------------------------------------------------------------------
Total Cost           $           0.1184 $           0.1162        -1.8%
Avg Latency (s)                    8.38               7.84        -6.4%
Input Tokens                     29,415             30,183        +2.6%
Output Tokens                     2,009              1,712       -14.8%

Result: 1.8% cost reduction, 6.4% latency improvement


### Results Analysis

Restructuring the prompt and tuning a couple of model parameters tightens the agent's responses without changing what it can do. Read the columns by *direction*, not exact figures — your run will differ, but the shape holds.

- **Output tokens go down.** This is the reliable effect, and the point of the lab. A clear output contract (answer / category / confidence) plus `max_tokens=1024` keep replies focused instead of rambling. Output tokens are billed at a higher rate than input, so this is the lever that moves cost.
- **Input tokens stay flat (may even tick up).** The v2 prompt is *restructured, not shorter* — it still carries the troubleshooting method and four examples. Don't expect input savings here; that's what **prompt caching** delivers in Lab 03.
- **Cost and latency drift down, modestly.** They follow the output reduction, so how much they move depends on how verbose your baseline answers happened to be. On a small 5-prompt run the swing is easily a few percent either way — don't read a precise number into it.

**Key takeaways:**

- **Structure beats brevity.** A well-organized prompt (headers, numbered guidelines, worked examples) helps the model find the right instruction and answer concisely — which shows up as fewer *output* tokens.
- **An explicit output format improves consistency.** Telling the model the exact fields to return makes behavior predictable across requests, which also makes the responses easier to evaluate later (Lab 07).
- **Model parameters are free wins.** `max_tokens` and `temperature` cost nothing to set but bound output length and stabilize tone.

The honest framing: these are *quick wins*, not a dramatic cut. The big cost reductions come later — caching (Lab 03) on the input side, routing (Lab 04) by picking a cheaper model per query.

---

**Next:** in Lab 03 we tackle the *input* side with **prompt caching** — a technique that can dramatically cut cost when the same system prompt is reused across requests.

## Going further: prompt-level techniques (2023–2026)

The three changes above are the actual *quick wins* — minutes to apply. This last section is a map of where to go **when they're not enough**, staying purely in **prompt territory** (no infra, no extra services). These range from a one-line tweak to a multi-call pipeline, so they're not all "quick" — but each is still about *what you ask for*, not how you host it.

| Technique | Effort | The idea | Cost / quality effect | Source |
|---|---|---|---|---|
| **Chain-of-Draft** | Quick (one line) | "Think step by step, but **keep each step ≤ 5 words**." Concise reasoning instead of verbose chain-of-thought. | Matches CoT accuracy on ~7.6% of the tokens (≈80% fewer) — big latency + cost cut. | [Xu et al., 2025](https://arxiv.org/abs/2502.18600) ([code](https://github.com/sileix/chain-of-draft)) |
| **Verbalized Sampling** | Moderate (reshapes output) | Ask for **N responses with explicit probabilities** ("give 5, each with its probability"), then sample. Counters the "same safe answer every time" mode collapse. | Training-free, orthogonal to temperature; ~1.6–2.1× more diversity for creative/open-ended tasks. | [Zhang et al., 2025](https://arxiv.org/abs/2510.01171) ([code](https://github.com/CHATS-lab/verbalize-sampling)) |
| **Self-Refine** | Heavier (multi-call loop) | Have the model **critique its own draft, then revise** — generate → feedback → refine, same model, no training. | Costs extra calls (largely subsumed by today's reasoning models), but a solid prompt-only quality fallback. | [Madaan et al., 2023](https://arxiv.org/abs/2303.17651) ([site](https://selfrefine.info/)) |
| **Chain-of-Verification** | Heavier (multi-call pipeline) | Draft → **plan verification questions** → answer them independently → revise. Independent checks catch the draft's own errors. | ~4–5× a single call; reach for it on **fact-critical** answers where a wrong claim is expensive. | [Dhuliawala et al., 2023](https://arxiv.org/abs/2309.11495) |
| **CROP** *(2026)* | Heavier (offline optimization run) | **Cost-Regularized Optimization of Prompts** — automatic prompt optimization with a *length* penalty in the feedback loop, so the tuned prompt elicits concise answers. | ~80.6% fewer tokens on GSM8K/LogiQA/BBH with only nominal accuracy loss. | [Shah et al., 2026](https://arxiv.org/abs/2604.14214) |
| **Prompt-Level Distillation** *(2026)* | Heavier (needs a teacher model) | Extract a strong "teacher" model's reasoning patterns into a **structured instruction list in a small model's system prompt** — distillation without any fine-tuning. | Gemma-3 4B Macro-F1 jumped 57→90% / 67→83%, matching frontier quality at small-model cost. | [Badhe & Shah, 2026](https://arxiv.org/abs/2602.21103) |

**How to choose:** *Chain-of-Draft* / *CROP* when reasoning is verbose and you want it cheaper; *Verbalized Sampling* when outputs feel repetitive; *Self-Refine* / *Chain-of-Verification* when correctness matters more than the extra calls; *Prompt-Level Distillation* when you want a small, cheap model to punch above its weight. Whichever you reach for, the discipline is the same as this lab — change one thing, then check the trace.

> The lighter ones trade a line of prompt for tokens; the heavier ones trade extra calls or an offline run for quality. Measure before adopting. Bigger structural levers (caching, routing, tool loading) come in the next labs.

---

## Cleanup

To delete the agent deployed in this notebook, uncomment and run the following code.

In [13]:
# # Uncomment to delete resources created in this lab
# agentcore_runtime.destroy(delete_ecr_repo=True)
# print(f"Deleted agent and ECR repository: {agent_name}")